In [2]:
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,MinMaxScaler,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression,LinearRegression
import numpy as np 
import pandas as pd


In [3]:
room=pd.read_csv("Room_Dataset.csv")
section=pd.read_csv("Student_Sections_DATASET.csv")
subjects=pd.read_csv("Subjects_Dataset.csv")
teachers=pd.read_csv("Teachers_Dataset.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'Room_Dataset.csv'

In [181]:
room.drop(columns=['Floor','Room Number'],inplace=True)


In [182]:
room.sample(10)

,Room ID,Building,Type,Department
43,SVH-201,SVH,Classroom,CSE
23,ENG-404,ENG,Classroom,CSE / Engineering
48,SVH-206,SVH,Classroom,CSE
102,LAW-207,LAW,Classroom,Law
117,ADM-209,ADM,Classroom,General / Shared
73,IGSM-209,IGSM,Classroom,Management
46,SVH-204,SVH,Classroom,CSE
126,NR-PhysicsLab,NR,Lab,Science
123,GH-202,GH,Classroom,General
3,ENG-104,ENG,Classroom,CSE / Engineering


In [269]:
teachers.head(15)

,Teacher ID,Teacher Name,Department,Main Subject,Backup Subject,Max Classes/Week,Preferred Slots,Can Take Labs,Can Be Class Coordinator
0,T-CSE-001,Aditi Kumar,CSE / Engineering,AI Lab,Major Project I,12,No 1st Period,Yes,No
1,T-CSE-002,Ananya Sinha,CSE / Engineering,Internship/Training,Python Lab,16,Morning,No,Yes
2,T-CSE-003,Harsh Das,CSE / Engineering,Internship/Training,Computer Networks,18,Afternoon,No,Yes
3,T-CSE-004,Aditi Verma,CSE / Engineering,Programming in C,DBMS Lab,12,No 1st Period,Yes,Yes
4,T-CSE-005,Nidhi Verma,CSE / Engineering,IoT Lab,IoT Systems,16,No 1st Period,Yes,Yes
5,T-CSE-006,Diya Pandey,CSE / Engineering,IoT Lab,Cyber Security,14,Afternoon,Yes,Yes
6,T-CSE-007,Simran Singh,CSE / Engineering,OOP with Java,Computer Networks,18,Morning,No,No
7,T-CSE-008,Sahil Jain,CSE / Engineering,Machine Learning,Mobile App Development,18,Morning,No,No
8,T-CSE-009,Pooja Thakur,CSE / Engineering,Networking Lab,IoT Lab,12,Any,Yes,Yes
9,T-CSE-010,Priya Sharma,CSE / Engineering,Data Structures,Software Engineering,18,Afternoon,Yes,No


In [300]:
engineering_subjects = subjects[subjects["Department"].str.strip() == "CSE / Engineering"]

subjects

,Department,Subject Name,Subject Type,Credits
0,CSE / Engineering,Programming in C,Theory,4
1,CSE / Engineering,Data Structures,Theory,4
2,CSE / Engineering,Operating Systems,Theory,4
3,CSE / Engineering,Computer Networks,Theory,4
4,CSE / Engineering,DBMS,Theory,4
...,...,...,...,...
265,Biotechnology,Environmental Biotech Lab,Lab,2
266,Biotechnology,Major Research Project,Project,1
267,Biotechnology,Virology,Theory,4
268,Biotechnology,Bioremediation,Theory,4


In [185]:
section.head(3)

,Department,Duration (Years),Section_ID,Current Year / Semester,Strength,Program
0,School of Management,4,2BBA1,Year 2 / Sem 3,45,BBA
1,School of Management,4,4BBA1,Year 4 / Sem 7,40,BBA
2,School of Management,4,1BBA1,Year 1 / Sem 1,43,BBA


In [186]:
Day=["Monday","Tuesday","Wednesday","Thursday","Friday"]

slots={f"{day} - {i+8} to {i+9}" for day in Day for i in range(8)}
slots

{'Friday - 10 to 11',
 'Friday - 11 to 12',
 'Friday - 12 to 13',
 'Friday - 13 to 14',
 'Friday - 14 to 15',
 'Friday - 15 to 16',
 'Friday - 8 to 9',
 'Friday - 9 to 10',
 'Monday - 10 to 11',
 'Monday - 11 to 12',
 'Monday - 12 to 13',
 'Monday - 13 to 14',
 'Monday - 14 to 15',
 'Monday - 15 to 16',
 'Monday - 8 to 9',
 'Monday - 9 to 10',
 'Thursday - 10 to 11',
 'Thursday - 11 to 12',
 'Thursday - 12 to 13',
 'Thursday - 13 to 14',
 'Thursday - 14 to 15',
 'Thursday - 15 to 16',
 'Thursday - 8 to 9',
 'Thursday - 9 to 10',
 'Tuesday - 10 to 11',
 'Tuesday - 11 to 12',
 'Tuesday - 12 to 13',
 'Tuesday - 13 to 14',
 'Tuesday - 14 to 15',
 'Tuesday - 15 to 16',
 'Tuesday - 8 to 9',
 'Tuesday - 9 to 10',
 'Wednesday - 10 to 11',
 'Wednesday - 11 to 12',
 'Wednesday - 12 to 13',
 'Wednesday - 13 to 14',
 'Wednesday - 14 to 15',
 'Wednesday - 15 to 16',
 'Wednesday - 8 to 9',
 'Wednesday - 9 to 10'}

In [292]:
teacher_subject_map={}

for _i,row in teachers.iterrows():
    if str(row["Department"]).strip()=="CSE / Engineering":
        teach_id=str(row["Teacher ID"]).strip() 
        main_subject=str(row.get("Main Subject","")).strip()
        backup_subject=str(row.get("Backup Subject","")).strip()
        teacher_name=str(row.get("Teacher Name","")).strip()
        is_lab=str(row.get("Can Take Labs","")).strip().lower() in ["Yes","true",1]
        room_type="lab" if is_lab else "Classroom"
        subject=[]
        if main_subject and main_subject.lower() !="nan":
            subject.append(f" {main_subject}  ({teacher_name})")
        if backup_subject and backup_subject.lower() !="nan":
            subject.append(f"{backup_subject} ({teacher_name})")
        for subj in [main_subject,backup_subject]:
            if subj:
                if subj not in teacher_subject_map:
                    teacher_subject_map[subj] = set()
                teacher_subject_map[subj].add(room_type)
        teacher_subject_map[teach_id]=subject

teacher_subject_map

{'AI Lab': {'Classroom'},
 'Major Project I': {'Classroom'},
 'T-CSE-001': [' AI Lab  (Aditi Kumar)', 'Major Project I (Aditi Kumar)'],
 'Internship/Training': {'Classroom'},
 'Python Lab': {'Classroom'},
 'T-CSE-002': [' Internship/Training  (Ananya Sinha)',
  'Python Lab (Ananya Sinha)'],
 'Computer Networks': {'Classroom'},
 'T-CSE-003': [' Internship/Training  (Harsh Das)',
  'Computer Networks (Harsh Das)'],
 'Programming in C': {'Classroom'},
 'DBMS Lab': {'Classroom'},
 'T-CSE-004': [' Programming in C  (Aditi Verma)', 'DBMS Lab (Aditi Verma)'],
 'IoT Lab': {'Classroom'},
 'IoT Systems': {'Classroom'},
 'T-CSE-005': [' IoT Lab  (Nidhi Verma)', 'IoT Systems (Nidhi Verma)'],
 'Cyber Security': {'Classroom'},
 'T-CSE-006': [' IoT Lab  (Diya Pandey)', 'Cyber Security (Diya Pandey)'],
 'OOP with Java': {'Classroom'},
 'T-CSE-007': [' OOP with Java  (Simran Singh)',
  'Computer Networks (Simran Singh)'],
 'Machine Learning': {'Classroom'},
 'Mobile App Development': {'Classroom'},
 'T

In [ ]:
subject_room={}

for _,sub_row in engineering_subjects.iterrows(): 
    sub=str(sub_row.get("Subject Name","")).strip()
    classrom_type=str(sub_row.get("Subject Type","")).strip()
    # if (main_subject==sub or backup_subject==sub):
    subject_room[sub]=[classrom_type]
for subj, types in subject_room.items():
    if subj.strip() in ["Programming in C", "Data Structures", "Operating Systems",
    "DBMS", "Computer Networks", "Software Engineering",
    "Cloud Computing", "Cyber Security", "Machine Learning","AI Lab", "Python Lab", "ML Lab", "Networking Lab", "IoT Lab"]:
        types.extend(["Lab", "Theory"])
        subject_room[subj] = list(set(types))
            # if "Theory" not in types:
                # types.append("Theory")  
            # if "Lab"not in types:
                # types.append("Lab")  
#     if sub not in subject_room:
#         subject_room[sub]=set()
#     subject_room[sub].add(classrom_type)
subject_room = {k: list(v) for k, v in subject_room.items()}
subject_room

subject_room



{'Programming in C': ['Theory', 'Lab'],
 'Data Structures': ['Theory', 'Lab'],
 'Operating Systems': ['Theory', 'Lab'],
 'Computer Networks': ['Theory', 'Lab'],
 'DBMS': ['Theory', 'Lab'],
 'DBMS Lab': ['Lab'],
 'Python Lab': ['Theory', 'Lab'],
 'Mini Project I': ['Project'],
 'OOP with Java': ['Theory'],
 'Software Engineering': ['Theory', 'Lab'],
 'Discrete Mathematics': ['Theory'],
 'Algorithms': ['Theory'],
 'Web Technologies': ['Theory'],
 'Computer Architecture': ['Theory'],
 'Compiler Design': ['Theory'],
 'Machine Learning': ['Theory', 'Lab'],
 'ML Lab': ['Theory', 'Lab'],
 'Cloud Computing': ['Theory', 'Lab'],
 'AI Lab': ['Theory', 'Lab'],
 'Cyber Security': ['Theory', 'Lab'],
 'Mobile App Development': ['Theory'],
 'Flutter Lab': ['Lab'],
 'IoT Systems': ['Theory'],
 'IoT Lab': ['Theory', 'Lab'],
 'Major Project I': ['Project'],
 'Major Project II': ['Project'],
 'Internship/Training': ['Project'],
 'Database Security': ['Theory'],
 'Networking Lab': ['Theory', 'Lab'],
 'Data

yes
no
no
yes
yes
yes
no
no
yes
yes
